**Conjunto de Datos**

El dataset de precios de autos usados en el Reino Unido es una recopilación de informacion obtenida a traves de Kaggle, https://www.kaggle.com/datasets/muhammadawaistayyab/used-cars-prices-in-uk/data. Este conjunto de datos contiene de 3.685 datos, cada uno representa un listado de vehiculos unico, e incluye trece caracteristicas distintivas que ofrecen informacion.

In [ ]:
# Importa las librerias necesarias para analisis y visualizacion de datos

import numpy as np              # Operaciones numericas y manejo de arrays
import pandas as pd             # Manipulacion y analisis de datos (DataFrames, lectura/escritura)
from matplotlib import pyplot as plt  # Generacion de graficos
import seaborn as sns           # Visualizacion estadistica con graficos mas estilizados (usa matplotlib internamente)

In [ ]:
# Define la ruta del archivo CSV en Google Drive
ruta = '/content/drive/MyDrive/Analisis Datos - TD/Portafolio/Kaggle - Costo Autos UK/used_cars_UK.csv'

# Carga el archivo CSV en un DataFrame de pandas
# index_col=0 indica que la primera columna del archivo sera usada como indice y no como columna de datos
df = pd.read_csv(ruta, index_col=0)

# Muestra las primeras 5 filas del DataFrame para inspeccion inicial
df.head()

In [ ]:
# Muestra la dimension del DataFrame
df.shape

In [ ]:
# Columnas, tipo de datos y nulos para cada una de ellas
df.info()

In [ ]:
# Estadistica descriptiva para cada columna
df.describe()

In [ ]:
# Calcula la cantidad de valores nulos (NaN) por columna en el DataFrame
df.isnull().sum()

In [ ]:
# Imprime mensaje indicando el inicio de la visualizacion de valores faltantes
print("Mapa de calor para valores faltantes:\n")

# Crea una figura con tamaño de 10x4
plt.figure(figsize=(10, 4))

# Genera un mapa de calor que marca las posiciones de valores nulos (True) en el DataFrame
# cbar=False oculta la barra de colores, cmap="viridis" define la paleta de color
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")

# Añade un titulo descriptivo al grafico
plt.title("Valores faltantes por columna")

# Muestra el grafico en pantalla
plt.show()

In [ ]:
# Imputacion de valores faltantes en el DataFrame

# Para 'Doors' y 'Seats':
#   - fillna(median()) reemplaza los NaN con la mediana de la columna
#   - round() redondea el valor a entero
#   - astype('Int64') convierte a entero manteniendo soporte para valores nulos (tipo entero de pandas)
df['Doors'] = df['Doors'].fillna(df['Doors'].median()).round().astype('Int64')
df['Seats'] = df['Seats'].fillna(df['Seats'].median()).round().astype('Int64')

# Para 'Previous Owners':
#   - fillna(mean()) reemplaza los NaN con la media de la columna
df['Previous Owners'] = df['Previous Owners'].fillna(df['Previous Owners'].mean())

In [ ]:
# Verifica nuevamente si existen valores faltantes en el DataFrame
# Devuelve la cantidad de NaN por columna despues de la imputacion
df.isnull().sum()

In [ ]:
# Elimina columnas que no aportan al análisis o generan ruido
# 'Service history', 'title' y 'Registration_Year' se eliminan completamente del DataFrame
df = df.drop(columns=['Service history', 'title', 'Registration_Year'])

# Elimina filas donde 'Emission Class' tenga valores NaN
# subset indica que la busqueda de NaN se hace solo en esta columna
df = df.dropna(subset=['Emission Class'])

In [ ]:
# Limpia la columna 'Engine' para convertirla en numerica
# str.replace('L', '', regex=True) elimina la letra 'L' de cada valor (ej.: '2.0L' a '2.0')
# astype(float) convierte los valores resultantes a tipo numerico flotante
df['Engine'] = df['Engine'].str.replace('L', '', regex=True).astype(float)

# Reemplaza valores NaN en la columna 'Engine' con 0
df['Engine'] = df['Engine'].fillna(0)

In [ ]:
# Limpia la columna 'Emission Class' eliminando el texto 'Euro'
# str.replace('Euro', '', regex=True) quita el prefijo (ej.: 'Euro 6' a '6')
# astype(int) convierte los valores resultantes a enteros
df['Emission Class'] = df['Emission Class'].str.replace('Euro', '', regex=True).astype(int)

In [ ]:
# Importa LabelEncoder para convertir variables categoricas (texto) a valores numericos
from sklearn.preprocessing import LabelEncoder

# Crea una instancia de LabelEncoder
# Este codificador asigna un numero entero unico a cada categoria en la columna
label_encoder = LabelEncoder()

In [ ]:
# Define las columnas categoricas que seran codificadas a valores numericos
columnas_categoricas = ['Fuel type', 'Body type', 'Gearbox']

# Aplica la codificacion LabelEncoder a cada columna categorica
# fit_transform() ajusta el codificador y transforma los valores en numeros enteros
for column in columnas_categoricas:
    df[column] = label_encoder.fit_transform(df[column])

In [ ]:
# Se comprueba el paso de columnas codificadas
df.head()

In [ ]:
# Define las variables para el modelo

# x: variables predictoras (todas las columnas excepto 'Price')
x = df.drop('Price', axis=1)

# y: variable objetivo (columna 'Price')
y = df['Price']

In [ ]:
# Importa la funcion train_test_split para dividir los datos en conjuntos de entrenamiento y prueba
from sklearn.model_selection import train_test_split

In [ ]:
# Divide los datos en conjuntos de entrenamiento y prueba
# test_size=0.2, el 20% de los datos se destina a prueba
# random_state=42, asegura que la division sea reproducible
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [ ]:
# Importa la clase LinearRegression para crear y entrenar un modelo de regresion lineal
from sklearn.linear_model import LinearRegression

In [ ]:
# Crea una instancia del modelo de regresion lineal
model = LinearRegression()

# Entrena el modelo con los datos de entrenamiento
model.fit(x_train, y_train)

# Genera predicciones de precios usando los datos de prueba
y_pred = model.predict(x_test)

In [ ]:
# Importa metricas para evaluar el rendimiento del modelo
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Calcula metricas de evaluacion del modelo
mae = mean_absolute_error(y_test, y_pred)  # Error absoluto medio
mse = mean_squared_error(y_test, y_pred)   # Error cuadrático medio
rmse = np.sqrt(mse)                        # Raiz del error cuadratico medio
r2 = r2_score(y_test, y_pred)              # Coeficiente de determinación R cuadrado

# Muestra las metricas formateadas a 2 decimales
print(f"Error absoluto medio (MAE): {mae:.2f}")
print(f"Error cuadratico medio (MSE): {mse:.2f}")
print(f"Raiz del Error Cuadratico Medio (RMSE): {rmse:.2f}")
print(f"R cuadrado (R2): {r2:.2f}")

In [ ]:
# Crea figura para la visualizacion de resultados
plt.figure(figsize=(10, 6))

# Grafico de dispersion (valores reales vs predichos)
sns.scatterplot(
    x=y_test, y=y_pred,
    color='b', alpha=0.9, edgecolor='k', s=80
)

# Linea de regresion (tendencia entre valores reales y predichos)
sns.regplot(
    x=y_test, y=y_pred,
    scatter=False, color='r',
    line_kws={"color": "orange", "lw": 2}
)

# Etiquetas y titulo del grafico
plt.xlabel("Valores Actuales", fontsize=14)
plt.ylabel("Valores Predecidos", fontsize=14)
plt.title("Rendimiento del modelo - Valores Actuales vs. Predecidos", fontsize=16)

# Muestra el grafico en pantalla
plt.show()

**Conclusiones**

El modelo de regresion lineal explica un 67 % de la variabilidad de los precios de autos usados (R² = 0,67). En la práctica, su error absoluto medio de ≈ £1.880 y un desvío típico de ≈ £2.710 indican que, para autos que rondan las £10.000, la predicción suele quedar a pocos miles de libras del precio real.

Considernado esto como un punto de partida, demuestra dominio de limpieza de datos, transformacion de variables y evaluación de metricas, estos resultados dejan margen para mejoras como la regularización y estandarizacion de funciones y pipelines.